# Lexical of 20th century Odyssey translations (Part B): Etymologies


____________________________________________________
## **Road Map**

**I. Libraries, files, and paths**

**II. The Texts**

1. Bibliographic information about the translators  
2. The translators at a glance tokenwise   

**III. TTR Analysis**

1. All-in, straightforward model  
    a) TTR Computation  
    b) Shapiro-Wilk test to check for normality  
    c) One-wat ANOVA for overall differences  
    d) Pairwise t-test using Bonferroni coprrection  
    c) Meassuring effect size ussing Cohen's d  

2. Adaptive models  
    a) Mixed-Effects model: author fixed effect / book as random effect  
    b) Standardized TTR:   
    c) Moving-average TTR: translation as temporal change  

3. Supplement models  
    a) Lexical Density   
    b) Diachronic analysis  
    c) Semantic fields:  

**IV. Zipf's Law**

**V. TF-IDF**



**VI. Discussing Results**

In [1]:
import autotime # Provision for anxious people
%load_ext autotime

time: 162 µs (started: 2025-04-20 22:24:50 +02:00)


In [2]:
# ----------------------------------------------------------------------
# Baic Libraries
# ----------------------------------------------------------------------

import sys 
import os

import ast
from collections import Counter

import re
import nltk

import numpy as np
import pandas as pd

import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

import scipy.stats as stats
from itertools import combinations

time: 3.38 s (started: 2025-04-20 22:24:50 +02:00)


In [3]:
# ----------------------------------------------------------------------
# Personalized Visualization & Functions
# ---------------------------------------------------------------------- 

sys.path.append('/Users/debr/English-Homer/functions') 
import matplotlib.pyplot as plt
import seaborn as sns

import e_chroma as chroma # My Vizualization library
import e_plots as oz      # My custom plots library
import e_pandisplay as pan# My pandas display options

import e_nlp_ody as e     # Import my nlp functions

import warnings           # Nononsense provision
warnings.filterwarnings('ignore')


* Got some chroma in your soma, Oma!
	 »----> use chroma.save_figure(fig, 'my_plot')
Default output path: ./Homer_xplots/

* OZ is behind the curtain!
	 »----> use oz.<func>
	 »----> also, oz goes chroma (for styling)!

*Has Pan taken over?
✓ Pandas display set to e_pandisplay defaults!
	 »----> use pan.<func>

* The editor is in the house!
	 »----> use e.<func> e.g. nlp = e.NLPPipeline(language='english')

Stopwords customized:
  Added: {'mrs', 'been', "'", 'of', 'two', 'mr', 'were', "'and", 'six', 'nine', 'that', 'one', 'them', 'was', 'are', 'be', 'seven', 'upon', 'n', 'she', 'this', 'four', 'there', 'it', 'five', 'they', 'is', 'ten', 'he', 'three', 'eight', 'being', 'said'}
  Removed: {''}
  Total stopwords: 215
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'…', "'", '\\', ',\n        "\'",\n        ', '-', '—'}
  Punctuation to be removed: !"#$%&'()*+,,
        "'",
        -./:;<=>?@[\]^_`{|}~—…
time: 977 ms (started: 2025-04-20 22:24:53 +02:00)


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# ----------------------------------------------------------------------
# File management
# ----------------------------------------------------------------------

# TO UPDATE
nb_id = "lexical_B01"

output_path = f"./"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"./{output_path}/{nb_id}_plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)
chroma.set_output_path(output_path_plots)

Output path set to: ././/lexical_B01_plots/
time: 926 µs (started: 2025-04-20 22:24:54 +02:00)


In [5]:
# ----------------------------------------------------------------------
# Odysseys
# ----------------------------------------------------------------------

translators = ['AT_Murray', 'Fitzgerald', 'Lattimore', 'Fagles', 'Wilson', 'Green', 'Woolf']

dfs = []

for odyssey in translators:
    filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{odyssey}_DataFrame.csv"
    temp_df = pd.read_csv(filepath)  
    dfs.append(temp_df)  # Append it to the list

df = pd.concat(dfs, axis=0, ignore_index=True)

df["text"] = df["text"].apply(ast.literal_eval)
df["tokens"] = df["tokens"].apply(ast.literal_eval)
df['translator'] = pd.Categorical(df['author'])
df["book_num"] = pd.Categorical(df["book_num"])
df = df[['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens']]

# ----------------------------------------------------------------------
# Backup dataframe only 'translator', 'book_num', 'text', 'tokens', columns
# ----------------------------------------------------------------------
df_bkp = df[['translator', 'book_num', 'text', 'tokens']].copy()
# ----------------------------------------------------------------------
# Dataframe check
# ----------------------------------------------------------------------

e.check_df(df)

Mr righteous here has no missing values!

* df columns: Index(['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens'], dtype='object') 

* Shape: (168, 6) 

* Total memory in MB: 4.064749
time: 776 ms (started: 2025-04-20 22:25:45 +02:00)
